# ChronoState-KD Lite — Phase 2: Distil the adapters

**Goal:** train three LoRA adapters on the 0.5B student from the cached teacher logits:

| Run | Domain | Config | Purpose |
| --- | --- | --- | --- |
| `adapter-code` | code | `lite.yaml` (α=0.7) | KD code adapter for serving |
| `adapter-reasoning` | reasoning | `lite.yaml` (α=0.7) | KD reasoning adapter for serving |
| `adapter-reasoning-sft` | reasoning | `lite_sft.yaml` (α=0) | **ablation**: same data, steps, seed, LR — no KL term |

**Budget:** ~25 min per run on a T4 (≈70 steps for code, ≈94 for reasoning at batch 2 × accum 8), ~80 min total. Each run checkpoints to Drive every 50 steps and resumes if the session dies; a finished run is skipped on re-run.

**Gate to pass before Phase 3:** three `adapters/<run>/adapter_model.safetensors` on Drive (~9 MB each); CE *and* KL fall over training in the KD runs; `train_summary.json` per run.

Runtime → **T4 GPU**. Edit `GITHUB_URL`. If you only have time for one run now, set `RUNS` to a single entry — order does not matter.

In [ ]:
GITHUB_URL = "https://github.com/<YOUR_GITHUB_USER>/chronostate-kd.git"   # <-- edit
REPO_DIR   = "/content/chronostate-kd"
DRIVE_ROOT = "/content/drive/MyDrive/chronostate"
RUNS = [  # (run_name, domain, config)
    ("adapter-code",          "code",      "configs/lite.yaml"),
    ("adapter-reasoning",     "reasoning", "configs/lite.yaml"),
    ("adapter-reasoning-sft", "reasoning", "configs/lite_sft.yaml"),
]


## 1. Setup

In [ ]:
import os, json, time, gc, torch
assert torch.cuda.is_available(), "Runtime -> Change runtime type -> T4 GPU"
print(torch.cuda.get_device_name(0))
if not os.path.exists(REPO_DIR):
    !git clone $GITHUB_URL $REPO_DIR
%cd $REPO_DIR
!git pull --ff-only
!pip install -q -r requirements.txt && pip install -q -e .
from google.colab import drive
drive.mount("/content/drive")
assert os.path.exists(f"{DRIVE_ROOT}/data/code.pt") and os.path.exists(f"{DRIVE_ROOT}/teacher_cache/reasoning/shard-0002.safetensors"), "Phase 1 artefacts missing on Drive"
from chronostate.config import load_config, to_dict
from chronostate.monitor import Monitor
from chronostate.train import train_adapter, build_student, quick_generate
from chronostate.data import SYSTEM_PROMPTS
mon = Monitor(f"{DRIVE_ROOT}/results/events.jsonl", run="phase2-distill")


## 2. Train
The loop (`chronostate/train.py`) prints every 10 steps: `loss`, `ce`, `kl`, grad-norm, LR, tokens/s. In the KD runs **both `ce` and `kl` should trend down**; in the SFT run `kl` is 0 by construction.

If Colab disconnects: reopen, run cells 1–2 again. Finished runs are skipped; an unfinished one resumes from its last 50-step checkpoint.

In [ ]:
summaries = {}
for run_name, domain, config in RUNS:
    print("=" * 80); print(run_name, domain, config); print("=" * 80)
    cfg = load_config(config)
    t0 = time.time()
    summaries[run_name] = train_adapter(cfg, run_name, domain, DRIVE_ROOT, monitor=mon, resume=True)
    print(f"{run_name}: {(time.time()-t0)/60:.1f} min wall")
    gc.collect(); torch.cuda.empty_cache()


## 3. Loss curves
The plot for the write-up: CE and KL per step for every run, KD vs SFT on the same axes.

In [ ]:
import matplotlib.pyplot as plt
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for run_name, s in summaries.items():
    h = s["history"]; steps = [r["step"] for r in h]
    axes[0].plot(steps, [r["ce"] for r in h], label=f"{run_name} CE")
    if s["alpha"] > 0:
        axes[1].plot(steps, [r["kl"] for r in h], label=f"{run_name} KL (tau^2-scaled)")
axes[0].set_title("Cross-entropy on response tokens"); axes[0].set_xlabel("optimizer step"); axes[0].legend()
axes[1].set_title("KL(teacher || student) over teacher top-64"); axes[1].set_xlabel("optimizer step"); axes[1].legend()
os.makedirs("results", exist_ok=True)
plt.tight_layout(); plt.savefig("results/phase2_loss_curves.png", dpi=130); plt.show()


## 4. Smoke test each adapter
Not an evaluation (that is Phase 5) — just proof that each adapter changes the student's behaviour in the intended direction.

In [ ]:
from peft import PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
cfg = load_config("configs/lite.yaml")
bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4", bnb_4bit_use_double_quant=True, bnb_4bit_compute_dtype=torch.float16)
tok = AutoTokenizer.from_pretrained(cfg.models.student)
base = AutoModelForCausalLM.from_pretrained(cfg.models.student, quantization_config=bnb, attn_implementation="sdpa", dtype=torch.float16, device_map={"": 0})
probes = {"code": "Write a Python function that returns the n-th Fibonacci number.",
          "reasoning": "A shop sells pens at $3 each. Ravi buys 4 pens and pays with a $20 bill. How much change does he get?"}
print("### BASE (no adapter)")
for d, p in probes.items(): print(f"[{d}]", quick_generate(base, tok, p, SYSTEM_PROMPTS[d], 80), "\n")
for run_name, domain, _ in RUNS:
    m = PeftModel.from_pretrained(base, f"{DRIVE_ROOT}/adapters/{run_name}")
    print(f"### {run_name}")
    print(quick_generate(m, tok, probes[domain], SYSTEM_PROMPTS[domain], 80), "\n")
    m.unload(); del m


## 5. Save summaries for the repo

In [ ]:
slim = {k: {kk: vv for kk, vv in v.items() if kk != "history"} for k, v in summaries.items()}
json.dump(slim, open("results/phase2_train_summary.json", "w"), indent=2)
json.dump(summaries, open(f"{DRIVE_ROOT}/results/phase2_train_summary_full.json", "w"), indent=2)
mon.close()
print(json.dumps(slim, indent=2))
!ls -la $DRIVE_ROOT/adapters/*


## Gate checklist
- [ ] three adapters on Drive, `adapter_model.safetensors` ≈ 9 MB each
- [ ] KD runs: `ce` and `kl` both lower at the end than at step 1
- [ ] SFT run: `kl` = 0 throughout; its final `ce` is comparable to (likely a bit lower than) the KD reasoning run's `ce` — that is expected, SFT optimises CE only
- [ ] `results/phase2_train_summary.json` and `results/phase2_loss_curves.png` downloaded and committed

Report back: the three final `loss / ce / kl` lines, step counts, minutes per run, peak VRAM, and paste the smoke-test outputs.